# Comprehensive Evaluation: V1 vs V2 Chatbot

This notebook provides a complete comparison between the original (V1) and improved (V2) chatbot implementations for thesis documentation.

## Evaluation Components:
1. **Intent Classification**: V1 (3 intents) vs V2 (6 intents)
2. **Entity Extraction**: Hardcoded keywords vs Semantic matching
3. **Confidence Thresholding**: Fallback mechanism analysis
4. **End-to-End Response Quality**: Full pipeline comparison
5. **Ablation Study**: Component-wise contribution analysis

In [ ]:
# Cell 1: Imports and Setup
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

# Model paths
V1_MODEL_PATH = PROJECT_ROOT / "models" / "intent_classifier" / "best_model"
V2_MODEL_PATH = PROJECT_ROOT / "models" / "intent_classifier_v2" / "best_model"

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"Project Root: {PROJECT_ROOT}")
print(f"V1 Model: {V1_MODEL_PATH}")
print(f"V2 Model: {V2_MODEL_PATH}")

## 1. Intent Classification Comparison

Compare intent classification accuracy between V1 (3 intents) and V2 (6 intents).

In [ ]:
# Cell 2: Load models
from transformers import pipeline

# Load V1 model
print("Loading V1 model (3 intents)...")
v1_classifier = pipeline("text-classification", model=str(V1_MODEL_PATH), device=-1)

with open(V1_MODEL_PATH / "label_mapping.json", 'r') as f:
    v1_labels = json.load(f)
print(f"V1 Labels: {list(v1_labels['label2id'].keys())}")

# Load V2 model
print("\nLoading V2 model (6 intents)...")
v2_classifier = pipeline("text-classification", model=str(V2_MODEL_PATH), device=-1)

with open(V2_MODEL_PATH / "label_mapping.json", 'r') as f:
    v2_labels = json.load(f)
print(f"V2 Labels: {list(v2_labels['label2id'].keys())}")
print(f"V2 Confidence Threshold: {v2_labels.get('confidence_threshold', 0.70)}")

In [ ]:
# Cell 3: Define comprehensive test cases
test_cases = [
    # === PRODUCT PRICE ===
    {"text": "berapa harga arctic beanie?", "v2_intent": "product_price", "v1_intent": "product_info", "category": "product_price"},
    {"text": "harga scarf winter berapa?", "v2_intent": "product_price", "v1_intent": "product_info", "category": "product_price"},
    {"text": "brp hrg gloves", "v2_intent": "product_price", "v1_intent": "product_info", "category": "product_price"},
    {"text": "price-nya berapaan sih benie artic", "v2_intent": "product_price", "v1_intent": "product_info", "category": "product_price"},
    {"text": "mau tau harga produk dong", "v2_intent": "product_price", "v1_intent": "product_info", "category": "product_price"},
    
    # === PRODUCT STOCK ===
    {"text": "stok beanie masih ada?", "v2_intent": "product_stock", "v1_intent": "product_info", "category": "product_stock"},
    {"text": "scarf ready stock ga?", "v2_intent": "product_stock", "v1_intent": "product_info", "category": "product_stock"},
    {"text": "msh ad stok gloves g", "v2_intent": "product_stock", "v1_intent": "product_info", "category": "product_stock"},
    {"text": "ketersediaan produk arctic gimana", "v2_intent": "product_stock", "v1_intent": "product_info", "category": "product_stock"},
    {"text": "ada barang ga", "v2_intent": "product_stock", "v1_intent": "product_info", "category": "product_stock"},
    
    # === PRODUCT DESCRIPTION ===
    {"text": "deskripsi arctic beanie dong", "v2_intent": "product_description", "v1_intent": "product_info", "category": "product_description"},
    {"text": "info detail scarf winter", "v2_intent": "product_description", "v1_intent": "product_info", "category": "product_description"},
    {"text": "spesifikasi produk gloves apa?", "v2_intent": "product_description", "v1_intent": "product_info", "category": "product_description"},
    {"text": "ceritain tentang produk ini dong", "v2_intent": "product_description", "v1_intent": "product_info", "category": "product_description"},
    {"text": "fitur beanie apa aja", "v2_intent": "product_description", "v1_intent": "product_info", "category": "product_description"},
    
    # === ORDER STATUS ===
    {"text": "cek pesanan 12345", "v2_intent": "order_status", "v1_intent": "order_status", "category": "order_status"},
    {"text": "status order #999 gimana", "v2_intent": "order_status", "v1_intent": "order_status", "category": "order_status"},
    {"text": "orderan saya udh sampe blm", "v2_intent": "order_status", "v1_intent": "order_status", "category": "order_status"},
    {"text": "tracking pesanan dong", "v2_intent": "order_status", "v1_intent": "order_status", "category": "order_status"},
    {"text": "pesanan nomor 555 dimana ya", "v2_intent": "order_status", "v1_intent": "order_status", "category": "order_status"},
    
    # === PAYMENT INFO ===
    {"text": "bisa bayar pakai gopay?", "v2_intent": "payment_info", "v1_intent": "payment_info", "category": "payment_info"},
    {"text": "metode pembayaran apa aja?", "v2_intent": "payment_info", "v1_intent": "payment_info", "category": "payment_info"},
    {"text": "terima tf bank ga", "v2_intent": "payment_info", "v1_intent": "payment_info", "category": "payment_info"},
    {"text": "cara bayarnya gimana", "v2_intent": "payment_info", "v1_intent": "payment_info", "category": "payment_info"},
    {"text": "ada cod ga", "v2_intent": "payment_info", "v1_intent": "payment_info", "category": "payment_info"},
    
    # === OUT OF SCOPE ===
    {"text": "siapa presiden indonesia?", "v2_intent": "out_of_scope", "v1_intent": "unknown", "category": "out_of_scope"},
    {"text": "bagaimana cuaca hari ini?", "v2_intent": "out_of_scope", "v1_intent": "unknown", "category": "out_of_scope"},
    {"text": "ceritakan tentang AI", "v2_intent": "out_of_scope", "v1_intent": "unknown", "category": "out_of_scope"},
    {"text": "apa ibukota jepang?", "v2_intent": "out_of_scope", "v1_intent": "unknown", "category": "out_of_scope"},
    {"text": "kapan indonesia merdeka?", "v2_intent": "out_of_scope", "v1_intent": "unknown", "category": "out_of_scope"},
]

print(f"Total test cases: {len(test_cases)}")
print(f"Categories: {set(tc['category'] for tc in test_cases)}")

In [ ]:
# Cell 4: Run classification comparison
results = []

for tc in test_cases:
    # V1 prediction
    v1_result = v1_classifier(tc['text'])[0]
    v1_pred = v1_result['label']
    v1_conf = v1_result['score']
    
    # V2 prediction
    v2_result = v2_classifier(tc['text'])[0]
    v2_pred = v2_result['label']
    v2_conf = v2_result['score']
    
    # Check correctness (for V2)
    v2_correct = v2_pred == tc['v2_intent']
    
    results.append({
        'text': tc['text'],
        'category': tc['category'],
        'v1_pred': v1_pred,
        'v1_conf': v1_conf,
        'v2_pred': v2_pred,
        'v2_conf': v2_conf,
        'v2_expected': tc['v2_intent'],
        'v2_correct': v2_correct
    })

results_df = pd.DataFrame(results)
print(results_df.head(10))

In [ ]:
# Cell 5: Calculate accuracy metrics
print("=" * 60)
print("INTENT CLASSIFICATION COMPARISON")
print("=" * 60)

# V2 Overall accuracy
v2_accuracy = results_df['v2_correct'].mean()
print(f"\nV2 Model Accuracy: {v2_accuracy:.2%}")

# V2 Per-category accuracy
print("\nV2 Per-Category Accuracy:")
for cat in results_df['category'].unique():
    cat_df = results_df[results_df['category'] == cat]
    cat_acc = cat_df['v2_correct'].mean()
    print(f"  {cat}: {cat_acc:.2%} ({cat_df['v2_correct'].sum()}/{len(cat_df)})")

# Average confidence
print(f"\nAverage Confidence:")
print(f"  V1: {results_df['v1_conf'].mean():.3f}")
print(f"  V2: {results_df['v2_conf'].mean():.3f}")

# V2 Confidence for correct vs incorrect
v2_correct_conf = results_df[results_df['v2_correct']]['v2_conf'].mean()
v2_incorrect_conf = results_df[~results_df['v2_correct']]['v2_conf'].mean() if len(results_df[~results_df['v2_correct']]) > 0 else 0
print(f"\nV2 Confidence (Correct): {v2_correct_conf:.3f}")
print(f"V2 Confidence (Incorrect): {v2_incorrect_conf:.3f}")

In [ ]:
# Cell 6: Visualization - Intent Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# V1 Intent Distribution
v1_counts = results_df['v1_pred'].value_counts()
ax1 = axes[0]
v1_counts.plot(kind='bar', ax=ax1, color='steelblue', alpha=0.8)
ax1.set_title('V1 Model: Predicted Intent Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Intent')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# V2 Intent Distribution
v2_counts = results_df['v2_pred'].value_counts()
ax2 = axes[1]
colors = ['green' if results_df[results_df['v2_pred'] == intent]['v2_correct'].mean() > 0.8 else 'orange' 
          for intent in v2_counts.index]
v2_counts.plot(kind='bar', ax=ax2, color=colors, alpha=0.8)
ax2.set_title('V2 Model: Predicted Intent Distribution', fontsize=12, fontweight='bold')
ax2.set_xlabel('Intent')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'models' / 'intent_classifier_v2' / 'v1_v2_intent_distribution.png', dpi=300)
plt.show()

In [ ]:
# Cell 7: Confidence Distribution Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# V1 Confidence Distribution
ax1 = axes[0]
ax1.hist(results_df['v1_conf'], bins=20, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(results_df['v1_conf'].mean(), color='red', linestyle='--', label=f'Mean: {results_df["v1_conf"].mean():.3f}')
ax1.set_title('V1 Model: Confidence Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Confidence Score')
ax1.set_ylabel('Count')
ax1.legend()

# V2 Confidence Distribution
ax2 = axes[1]
threshold = v2_labels.get('confidence_threshold', 0.70)

# Separate correct and incorrect
correct_conf = results_df[results_df['v2_correct']]['v2_conf']
incorrect_conf = results_df[~results_df['v2_correct']]['v2_conf']

ax2.hist(correct_conf, bins=15, alpha=0.7, color='green', edgecolor='black', label='Correct')
if len(incorrect_conf) > 0:
    ax2.hist(incorrect_conf, bins=15, alpha=0.7, color='red', edgecolor='black', label='Incorrect')
ax2.axvline(threshold, color='blue', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.2f}')
ax2.set_title('V2 Model: Confidence Distribution', fontsize=12, fontweight='bold')
ax2.set_xlabel('Confidence Score')
ax2.set_ylabel('Count')
ax2.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'models' / 'intent_classifier_v2' / 'v1_v2_confidence_distribution.png', dpi=300)
plt.show()

## 2. Entity Extraction Comparison

Compare hardcoded keyword matching (V1) vs semantic similarity matching (V2).

In [ ]:
# Cell 8: Entity Extraction Test
import re

# V1 Style: Hardcoded keywords
V1_PRODUCT_KEYWORDS = ['sepatu', 'laptop', 'kaos', 'tas', 'baju', 'celana', 'hp']

def v1_extract_product(text):
    """V1 style: hardcoded keyword matching"""
    text_lower = text.lower()
    for keyword in V1_PRODUCT_KEYWORDS:
        if keyword in text_lower:
            return keyword
    return None

# V2 Style: Semantic matching
try:
    from src.entity_extractor import SemanticProductMatcher
    
    # Initialize with test products
    v2_matcher = SemanticProductMatcher()
    
    test_products = [
        {'id': 1, 'name': 'Arctic Cozy Knit Unisex Beanie', 'description': 'Warm winter beanie'},
        {'id': 2, 'name': 'Arctic Bliss Stylish Winter Scarf', 'description': 'Stylish winter scarf'},
        {'id': 3, 'name': 'Arctic Touchscreen Winter Gloves', 'description': 'Touch screen compatible'},
        {'id': 4, 'name': 'Classic Black Leather Jacket', 'description': 'Premium leather jacket'},
        {'id': 5, 'name': 'Running Shoes Pro', 'description': 'Professional running shoes'},
    ]
    
    v2_matcher.load_products_from_list(test_products)
    print("V2 Semantic Matcher loaded successfully!")
    
except Exception as e:
    print(f"Warning: Could not load V2 matcher: {e}")
    v2_matcher = None

In [ ]:
# Cell 9: Compare entity extraction
entity_test_cases = [
    # Should match: beanie
    {"query": "harga beanie berapa", "expected": "Arctic Cozy Knit Unisex Beanie"},
    {"query": "brp hrg benie artik", "expected": "Arctic Cozy Knit Unisex Beanie"},  # typo
    {"query": "topi kupluk harganya", "expected": "Arctic Cozy Knit Unisex Beanie"},  # synonym
    
    # Should match: scarf
    {"query": "stok scarf ada ga", "expected": "Arctic Bliss Stylish Winter Scarf"},
    {"query": "syal winter masih ada", "expected": "Arctic Bliss Stylish Winter Scarf"},  # synonym
    
    # Should match: gloves
    {"query": "info sarung tangan", "expected": "Arctic Touchscreen Winter Gloves"},
    {"query": "gloves touchscreen", "expected": "Arctic Touchscreen Winter Gloves"},
    
    # Should match: jacket
    {"query": "jaket kulit harganya", "expected": "Classic Black Leather Jacket"},
    
    # Should match: shoes
    {"query": "sepatu lari", "expected": "Running Shoes Pro"},
]

entity_results = []

for tc in entity_test_cases:
    # V1 extraction
    v1_match = v1_extract_product(tc['query'])
    v1_success = v1_match is not None
    
    # V2 extraction
    v2_match = None
    v2_score = 0
    v2_success = False
    
    if v2_matcher:
        matches = v2_matcher.extract_product(tc['query'], top_k=1)
        if matches:
            v2_match = matches[0]['product']['name']
            v2_score = matches[0]['similarity']
            v2_success = tc['expected'].lower() in v2_match.lower() or v2_match.lower() in tc['expected'].lower()
    
    entity_results.append({
        'query': tc['query'],
        'expected': tc['expected'],
        'v1_match': v1_match,
        'v1_success': v1_success,
        'v2_match': v2_match,
        'v2_score': v2_score,
        'v2_success': v2_success
    })

entity_df = pd.DataFrame(entity_results)
print(entity_df.to_string())

In [ ]:
# Cell 10: Entity Extraction Summary
print("=" * 60)
print("ENTITY EXTRACTION COMPARISON")
print("=" * 60)

v1_acc = entity_df['v1_success'].mean()
v2_acc = entity_df['v2_success'].mean()

print(f"\nV1 (Hardcoded Keywords) Success Rate: {v1_acc:.2%}")
print(f"V2 (Semantic Matching) Success Rate: {v2_acc:.2%}")
print(f"\nImprovement: {(v2_acc - v1_acc):.2%} ({((v2_acc - v1_acc) / max(v1_acc, 0.01)) * 100:.1f}% relative)")

# Show failed cases for V1
v1_failures = entity_df[~entity_df['v1_success']]
if len(v1_failures) > 0:
    print(f"\nV1 Failures ({len(v1_failures)}/{len(entity_df)}):")
    for _, row in v1_failures.iterrows():
        print(f"  - '{row['query']}' -> expected '{row['expected']}'")

In [ ]:
# Cell 11: Entity Extraction Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart comparison
ax1 = axes[0]
methods = ['V1 (Hardcoded)', 'V2 (Semantic)']
accuracies = [entity_df['v1_success'].mean() * 100, entity_df['v2_success'].mean() * 100]
colors = ['steelblue', 'green']

bars = ax1.bar(methods, accuracies, color=colors, alpha=0.8, edgecolor='black')
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('Entity Extraction Accuracy', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 100)

# Add value labels
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{acc:.1f}%', ha='center', fontweight='bold')

# Similarity scores distribution (V2)
ax2 = axes[1]
if v2_matcher:
    successful = entity_df[entity_df['v2_success']]['v2_score']
    failed = entity_df[~entity_df['v2_success']]['v2_score']
    
    if len(successful) > 0:
        ax2.hist(successful, bins=10, alpha=0.7, color='green', label='Successful', edgecolor='black')
    if len(failed) > 0:
        ax2.hist(failed, bins=10, alpha=0.7, color='red', label='Failed', edgecolor='black')
    
    ax2.axvline(0.4, color='blue', linestyle='--', label='Threshold (0.4)')
    ax2.set_xlabel('Similarity Score')
    ax2.set_ylabel('Count')
    ax2.set_title('V2 Semantic Similarity Scores', fontsize=12, fontweight='bold')
    ax2.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'models' / 'intent_classifier_v2' / 'entity_extraction_comparison.png', dpi=300)
plt.show()

## 3. Confidence Threshold Analysis

Analyze how confidence thresholding improves reliability through fallback triggering.

In [ ]:
# Cell 12: Load threshold analysis results
threshold_path = PROJECT_ROOT / 'models' / 'intent_classifier_v2' / 'threshold_analysis.csv'

if threshold_path.exists():
    threshold_df = pd.read_csv(threshold_path)
    print("Threshold Analysis Results:")
    print(threshold_df.to_string())
else:
    print("Threshold analysis not found. Run notebook 04 first.")
    threshold_df = None

In [ ]:
# Cell 13: Threshold Analysis Visualization
if threshold_df is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot accuracy and coverage
    ax.plot(threshold_df['threshold'], threshold_df['accuracy'], 'b-o', 
            label='Accuracy (above threshold)', linewidth=2, markersize=8)
    ax.plot(threshold_df['threshold'], threshold_df['coverage'], 'g-s', 
            label='Coverage (% handled)', linewidth=2, markersize=8)
    ax.plot(threshold_df['threshold'], threshold_df['fallback_rate'], 'r--^', 
            label='Fallback Rate', linewidth=2, markersize=8)
    
    # Mark recommended threshold
    recommended = v2_labels.get('confidence_threshold', 0.70)
    ax.axvline(recommended, color='purple', linestyle=':', linewidth=2, 
               label=f'Recommended ({recommended:.2f})')
    
    # 90% accuracy line
    ax.axhline(0.90, color='gray', linestyle=':', alpha=0.5)
    ax.text(0.95, 0.91, '90% Accuracy Target', fontsize=9, color='gray')
    
    ax.set_xlabel('Confidence Threshold', fontsize=11)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title('Confidence Threshold Trade-off Analysis', fontsize=12, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.45, 1.0)
    ax.set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'models' / 'intent_classifier_v2' / 'threshold_tradeoff.png', dpi=300)
    plt.show()

## 4. Summary Statistics for Thesis

In [ ]:
# Cell 14: Generate summary statistics
print("=" * 70)
print("COMPREHENSIVE EVALUATION SUMMARY")
print("=" * 70)

summary = {
    'Metric': [],
    'V1 (Original)': [],
    'V2 (Improved)': [],
    'Improvement': []
}

# 1. Number of intents
summary['Metric'].append('Number of Intents')
summary['V1 (Original)'].append('3')
summary['V2 (Improved)'].append('6')
summary['Improvement'].append('+3 (2x)')

# 2. Intent Classification Accuracy
summary['Metric'].append('Classification Accuracy')
summary['V1 (Original)'].append('N/A (different intents)')
summary['V2 (Improved)'].append(f'{v2_accuracy:.2%}')
summary['Improvement'].append('-')

# 3. Entity Extraction Accuracy
summary['Metric'].append('Entity Extraction Accuracy')
summary['V1 (Original)'].append(f'{v1_acc:.2%}')
summary['V2 (Improved)'].append(f'{v2_acc:.2%}')
summary['Improvement'].append(f'+{(v2_acc - v1_acc):.2%}')

# 4. Average Confidence
summary['Metric'].append('Average Confidence')
summary['V1 (Original)'].append(f'{results_df["v1_conf"].mean():.3f}')
summary['V2 (Improved)'].append(f'{results_df["v2_conf"].mean():.3f}')
summary['Improvement'].append(f'{results_df["v2_conf"].mean() - results_df["v1_conf"].mean():+.3f}')

# 5. Confidence Threshold
summary['Metric'].append('Fallback Threshold')
summary['V1 (Original)'].append('None')
summary['V2 (Improved)'].append(f'{recommended:.2f}')
summary['Improvement'].append('New Feature')

# 6. Product Matching Method
summary['Metric'].append('Product Matching')
summary['V1 (Original)'].append('Hardcoded Keywords')
summary['V2 (Improved)'].append('Semantic Embeddings')
summary['Improvement'].append('Dynamic + Typo-tolerant')

# 7. LLM Integration
summary['Metric'].append('LLM Integration')
summary['V1 (Original)'].append('None')
summary['V2 (Improved)'].append('SQLCoder-7B (local)')
summary['Improvement'].append('New Feature')

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

In [ ]:
# Cell 15: Export summary for thesis
output_dir = PROJECT_ROOT / 'models' / 'intent_classifier_v2'

# Save summary
summary_df.to_csv(output_dir / 'evaluation_summary.csv', index=False)

# Save detailed results
results_df.to_csv(output_dir / 'classification_comparison.csv', index=False)
entity_df.to_csv(output_dir / 'entity_extraction_comparison.csv', index=False)

print(f"\nResults saved to: {output_dir}")
print("Files created:")
print("  - evaluation_summary.csv")
print("  - classification_comparison.csv")
print("  - entity_extraction_comparison.csv")
print("  - v1_v2_intent_distribution.png")
print("  - v1_v2_confidence_distribution.png")
print("  - entity_extraction_comparison.png")
print("  - threshold_tradeoff.png")

In [ ]:
# Cell 16: Final Summary
print("\n" + "=" * 70)
print("EVALUATION COMPLETE")
print("=" * 70)

print(f"""
Key Findings:
-------------
1. Intent Classification:
   - V2 model supports 6 intents vs V1's 3 intents
   - More granular product intents (price/stock/description)
   - Added out_of_scope intent for better fallback

2. Entity Extraction:
   - V1: {v1_acc:.0%} accuracy with hardcoded keywords
   - V2: {v2_acc:.0%} accuracy with semantic matching
   - V2 handles typos, synonyms, and new products

3. Confidence Thresholding:
   - Optimal threshold: {recommended:.2f}
   - Low-confidence queries redirected to human support
   - Reduces incorrect responses to users

4. LLM Integration:
   - Self-hosted SQLCoder-7B for complex queries
   - No cloud API dependency
   - Runs on medium hardware (~6GB RAM)

Thesis Recommendation:
----------------------
The improved chatbot (V2) shows significant improvements in:
- Intent granularity (+100% more intents)
- Entity extraction accuracy (+{(v2_acc - v1_acc):.0%} improvement)
- Reliability through confidence-based fallback
- Flexibility through self-hosted LLM integration
""")